# 135 -- DeerFlow Research Skill

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/135-deerflow-research-skill/deerflow_research_skill_workbook.ipynb)

## What you will learn

- What a DeerFlow skill is: a `SKILL.md` file composed of YAML front-matter and a markdown system prompt
- The four required front-matter fields: `name`, `trigger`, `description` (optional extras: `input_format`, `output_format`)
- How DeerFlow's trigger dispatcher detects `@`-mentions in messages and routes them to the registered skill
- How `subagent_enabled=True` causes DeerFlow to act on delegation instructions written in the skill body
- How skills differ from tools: skills are prompt-level compositions; tools are code-level Python functions
- The full invocation flow: upload files -> send `@skill-name` prompt -> stream SSE events -> receive artifact

**Source:** `examples/135-deerflow-research-skill/`

---

## Notebook structure

| Part | Requires | What it covers |
|------|----------|----------------|
| A | CPU only | Read the real SKILL.md from disk, parse front-matter, simulate dispatch, exercises |
| B | DeerFlow at `localhost:8000` | Upload files, invoke `@course-research`, stream live events, inspect artifact |

## Where this example fits

| Example | Pattern | What it adds |
|---------|---------|-------------|
| 134 -- DeerFlow Embedded Client | Raw HTTP + SSE streaming | Baseline: talk to DeerFlow with plain `httpx` |
| **135 -- DeerFlow Research Skill** | **Custom skill registration** | **SKILL.md format, trigger dispatch, subagent delegation** |
| 136 -- SWE Agent | Autonomous coding agent | Multi-turn tool use for software engineering tasks |

In [ ]:
%pip install -q httpx python-dotenv

## Part A -- CPU-Safe Concept Demonstrations

All cells in Part A run entirely on the local machine. No server is needed.
They read the real `SKILL.md` from disk and simulate the behaviour DeerFlow
performs at runtime.

### A1 -- What a DeerFlow Skill Is

A DeerFlow skill is a directory that contains exactly one `SKILL.md` file.
DeerFlow scans the paths listed under `skills.custom` in `conf/config.yaml`
at startup, reads the YAML front-matter to register the skill in its trigger
registry, and injects the markdown body as a system-prompt override whenever
the skill is activated.

**Directory layout**

```
deer-flow/
  conf/
    config.yaml          <- lists skills.custom paths
  skills/
    custom/
      course-research/
        SKILL.md         <- the skill definition
```

**config.yaml reference**

```yaml
skills:
  custom:
    - path: skills/custom/course-research
      enabled: true
```

> The markdown body is the actual prompt -- DeerFlow does NOT parse it for
> logic. The LLM reads the markdown and decides what actions to take.
> This means you can iterate on skill behaviour by editing plain text,
> with no Python changes required.

### A2 -- The SKILL.md Format

A `SKILL.md` file has two parts separated by `---` delimiters:

1. **YAML front-matter** (between the `---` delimiters): declares metadata
   DeerFlow reads at startup.
2. **Markdown body**: the system prompt injected into the conversation when
   the skill fires.

**Front-matter field reference**

| Field | Required | Purpose | Example |
|-------|----------|---------|--------|
| `name` | yes | Internal identifier used in config.yaml | `course-research` |
| `trigger` | yes | The `@`-mention that activates the skill | `@course-research` |
| `description` | yes | Shown in skill listings and plan output | `Generate a structured research report...` |
| `input_format` | no | Hint to the LLM about expected input shape | `markdown files` |
| `output_format` | no | Hint to the LLM about expected output shape | `markdown artifact` |

**Annotated example (the real course-research skill)**

```markdown
---
name: course-research          # internal ID, matches config.yaml path
trigger: "@course-research"    # @-token that fires this skill
description: >                 # multiline YAML scalar
  Generate a structured research report from uploaded thread materials.
---

# Course Research Skill        <- body starts here; this is the system prompt

You are a research assistant for the agent-framework course.
...
```

The `---` delimiter closes the front-matter block. Everything after it is the
markdown body that DeerFlow injects verbatim as a system prompt.

In [ ]:
import os
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())

SKILL_MD_CANDIDATES = [
    NOTEBOOK_DIR / "runtime" / "skills" / "custom" / "course-research" / "SKILL.md",
    NOTEBOOK_DIR / "examples" / "135-deerflow-research-skill" / "runtime" / "skills" / "custom" / "course-research" / "SKILL.md",
]

skill_path = next((p for p in SKILL_MD_CANDIDATES if p.exists()), None)

FALLBACK_SKILL = """---
name: course-research
trigger: \"@course-research\"
description: >
  Generate a structured research report from uploaded thread materials.
---

# Course Research Skill

You are a research assistant for the agent-framework course.

When activated with `@course-research <query>`:

1. Read all markdown files uploaded to the current thread workspace.
2. Identify key concepts, implementation patterns, and design trade-offs.
3. Generate a structured report with these sections:
   - **Executive Summary** (2-3 sentences)
   - **Pattern Comparison** (table: Pattern | When to Use | Trade-offs)
   - **Implementation Notes** (bullet list of actionable details)
   - **Next Steps** (2-3 suggested follow-up examples from the course)

Write the final report as a markdown artifact named `research-report.md`.

Delegate section drafting to subagents when multiple source files are present:
- Subagent A: summarise patterns from all uploaded files
- Subagent B: draft the comparison table
- Supervisor: merge and finalise into `research-report.md`
"""

if skill_path:
    SKILL_CONTENT = skill_path.read_text()
    print(f"Read from: {skill_path}")
else:
    SKILL_CONTENT = FALLBACK_SKILL
    print("[SKILL.md not found at expected path -- using embedded copy]")

print("\n" + "=" * 60)
print(SKILL_CONTENT)
print("=" * 60)


def parse_skill_frontmatter(content):
    """Extract name/trigger/description from YAML front-matter without a YAML library."""
    result = {}
    lines = content.splitlines()
    delims = [i for i, l in enumerate(lines) if l.strip() == "---"]
    if len(delims) < 2:
        return result
    current_key = ""
    for line in lines[delims[0] + 1 : delims[1]]:
        if ": " in line and not line.startswith(" "):
            key, _, val = line.partition(": ")
            current_key = key.strip()
            result[current_key] = val.strip().strip('"')
        elif line.startswith("  ") and current_key:
            result[current_key] = (result.get(current_key, "") + " " + line.strip()).strip()
    return result


def extract_skill_body(content):
    """Return the markdown body (everything after the closing --- delimiter)."""
    lines = content.splitlines()
    delims = [i for i, l in enumerate(lines) if l.strip() == "---"]
    if len(delims) < 2:
        return content
    return "\n".join(lines[delims[1] + 1 :]).strip()


meta = parse_skill_frontmatter(SKILL_CONTENT)
body = extract_skill_body(SKILL_CONTENT)

print("\nParsed front-matter:")
for k, v in meta.items():
    print(f"  {k:<14} = {v!r}")
print(f"\nBody: {len(body)} chars, starts with: {body[:60]!r}")

### A3 -- How DeerFlow Routes Skill Triggers

When a chat message arrives, DeerFlow runs a simple trigger-dispatch pipeline
before deciding whether to use a plain LLM call or a skill-injected call:

```
User message arrives
        |
        v
 Tokenize on whitespace
        |
        v
 For each token: is it in the skill registry?
     yes |                    no |
         v                       v
   Extract skill             Plain chat
   Strip trigger             -> LLM directly
   from message
         |
         v
   Inject skill body
   as system prompt
         |
         v
   subagent_enabled?
     yes |         no |
         v             v
   Spawn workers    Single LLM
   per body         handles all
   instructions
```

> **Key insight:** the trigger is just an `@`-token match against the registry.
> DeerFlow does NOT parse the skill body for logic -- the LLM reads the
> markdown instructions and decides what to do. This means you can write
> delegation, branching, and tool-use patterns in plain English.

In [ ]:
# Build a minimal skill registry from the parsed front-matter
meta = parse_skill_frontmatter(SKILL_CONTENT)
body = extract_skill_body(SKILL_CONTENT)

SKILL_REGISTRY = {
    meta.get("trigger", "@course-research"): {
        "name": meta.get("name", "course-research"),
        "body": body,
        "description": meta.get("description", ""),
    }
}


def dispatch_skill(message, registry):
    """Return (skill_name, cleaned_message) if a trigger found, else (None, message)."""
    tokens = message.split()
    for token in tokens:
        if token in registry:
            skill = registry[token]
            cleaned = " ".join(t for t in tokens if t != token).strip()
            return skill["name"], cleaned
    return None, message


test_messages = [
    "@course-research Generate a structured research report.",
    "@course-research Summarise the uploaded notes.",
    "What is the reactive loop pattern?",
    "@nonexistent-skill Do something.",
]

print("Trigger dispatch simulation:\n")
for msg in test_messages:
    skill_name, cleaned = dispatch_skill(msg, SKILL_REGISTRY)
    if skill_name:
        print(f"  INPUT : {msg!r}")
        print(f"  SKILL : {skill_name}")
        print(f"  QUERY : {cleaned!r}")
    else:
        print(f"  INPUT : {msg!r}")
        print(f"  SKILL : (none -- plain chat)")
    print()

### A4 -- Skills vs Tools: What is the Difference?

DeerFlow supports two extension mechanisms. Understanding when to use each
is important for designing agent workflows.

| Dimension | Skill | Tool |
|-----------|-------|------|
| Definition | `SKILL.md` file with a prompt template | Python function with `@tool` decorator |
| Invocation | `@`-mention in message text | LLM generates a `tool_call` JSON block |
| Execution | LLM follows prompt instructions | Runtime calls the Python function |
| Extensibility | Edit the markdown body | Write Python code |
| Subagent support | Yes (via delegation instructions in body) | No |
| Output format | Free-form text or markdown artifact | Structured return value |
| Best for | Research, synthesis, multi-step reasoning | Web search, file I/O, code execution, APIs |

> Skills and tools **compose well**. A skill prompt can instruct the LLM to
> call built-in tools (`web_search`, `file_read`, `file_write`) as part of
> its execution. The skill defines the high-level workflow; tools handle
> atomic actions.

In [ ]:
# Show the difference between a skill event trace and a direct tool invocation

# When a skill fires, the first event DeerFlow emits includes the injected system prompt.
# A direct tool call does NOT have this -- it goes straight to a tool_call event.

SKILL_TRACE = [
    ("skill_activated", {"skill": "course-research", "trigger": "@course-research"}),
    ("tool_call",       {"tool_name": "file_read", "args": {"filename": "01-state-graph.md"}}),
    ("tool_result",     {"content": "# State Graph Pattern\nExplicit directed graph..."}),
    ("message_chunk",   {"content": "## Executive Summary\n"}),
    ("end",             {}),
]

DIRECT_TOOL_TRACE = [
    ("tool_call",     {"tool_name": "web_search", "args": {"query": "state graph pattern LangGraph"}}),
    ("tool_result",   {"content": "LangGraph StateGraph: nodes process state..."}),
    ("message_chunk", {"content": "The state graph pattern uses..."}),
    ("end",           {}),
]

_ICONS = {
    "skill_activated": "[skill] ",
    "tool_call":       "[tool]  ",
    "tool_result":     "[result]",
    "message_chunk":   "[chunk] ",
    "end":             "[end]   ",
}


def print_trace(label, trace):
    print(f"  {label}")
    for et, data in trace:
        icon = _ICONS.get(et, "[?]    ")
        if et == "skill_activated":
            print(f"    {icon} skill={data['skill']}  trigger={data['trigger']}")
        elif et == "tool_call":
            print(f"    {icon} {data['tool_name']}({list(data['args'].keys())})")
        elif et == "tool_result":
            print(f"    {icon} {data['content'][:50]!r}")
        elif et == "message_chunk":
            print(f"    {icon} {data['content']!r}")
        elif et == "end":
            print(f"    {icon}")
    print()


print("Event trace comparison:\n")
print_trace("Skill invocation (@course-research ...)", SKILL_TRACE)
print_trace("Direct tool invocation (no skill trigger)", DIRECT_TOOL_TRACE)
print("Notice: the skill trace starts with skill_activated before any tool calls.")
print("The direct trace goes straight to tool_call -- no system prompt injection.")

In [ ]:
import json

# Show the registration request shape and the trigger prompt
skill_upload_payload = {
    "method": "POST",
    "url": "/api/files/upload",
    "form_data": {
        "file": ("SKILL.md", SKILL_CONTENT.encode(), "text/markdown"),
        "thread_id": "skill-setup-thread",
    },
}

print("Skill registration request shape:")
print(f"  Method   : {skill_upload_payload['method']}")
print(f"  URL      : {skill_upload_payload['url']}")
print(f"  file     : SKILL.md ({len(SKILL_CONTENT)} bytes, text/markdown)")
print(f"  thread_id: {skill_upload_payload['form_data']['thread_id']}")
print()

trigger_prompt = (
    "@course-research Generate a structured research report: "
    "executive summary, pattern comparison table, implementation notes."
)
print("Trigger prompt:")
print(f"  {trigger_prompt!r}")
print()

chat_body = {
    "message": trigger_prompt,
    "thread_id": "research-thread-001",
    "plan_mode": True,
    "subagent_enabled": True,
}
print("Chat request body (POST /api/chat/stream):")
print(json.dumps(chat_body, indent=2))

In [ ]:
# Mock skill execution -- simulate what the server event stream would look like

MOCK_SKILL_EVENTS = [
    ("tool_call",     {"tool_name": "plan", "args": {"steps": ["read uploaded files", "identify patterns", "draft report sections", "merge into artifact"]}}),
    ("tool_result",   {"content": "Plan confirmed: 4 steps."}),
    ("tool_call",     {"tool_name": "file_read", "args": {"filename": "01-state-graph.md"}}),
    ("tool_result",   {"content": "# State Graph Pattern\nExplicit directed graph..."}),
    ("tool_call",     {"tool_name": "file_read", "args": {"filename": "02-reactive-loop.md"}}),
    ("tool_result",   {"content": "# Reactive Loop Pattern\nAgent calls tools..."}),
    ("tool_call",     {"tool_name": "file_read", "args": {"filename": "03-multi-agent.md"}}),
    ("tool_result",   {"content": "# Multi-Agent Pattern\nSupervisor delegates..."}),
    ("message_chunk", {"content": "## Executive Summary\n"}),
    ("message_chunk", {"content": "Three agent patterns cover most LLM workflows..."}),
    ("tool_call",     {"tool_name": "file_write", "args": {"filename": "research-report.md", "content": "..."}}),
    ("tool_result",   {"content": "Artifact written: research-report.md"}),
    ("end",           {}),
]

_MOCK_ICONS = {
    "tool_call":     "[tool]  ",
    "tool_result":   "[result]",
    "message_chunk": "[chunk] ",
    "end":           "[end]   ",
}

print("Mock skill execution trace (plan_mode=True, subagent_enabled=True):\n")
for et, data in MOCK_SKILL_EVENTS:
    icon = _MOCK_ICONS.get(et, "[?]    ")
    if et == "tool_call":
        print(f"  {icon} {data['tool_name']}  args={list(data['args'].keys())}")
    elif et == "tool_result":
        print(f"  {icon} {data['content'][:60]!r}")
    elif et == "message_chunk":
        print(f"  {icon} {data['content']!r}")
    elif et == "end":
        print(f"  {icon}")

print()
counts = {}
for et, _ in MOCK_SKILL_EVENTS:
    counts[et] = counts.get(et, 0) + 1
print("Event summary:", {k: v for k, v in sorted(counts.items())})

### A5 -- When Does a Skill Add Value?

Adding a skill has overhead (a config change, a file to maintain). It is worth
it when the workflow is recurring and needs consistent output.

| Scenario | Use Skill? | Reason |
|----------|------------|--------|
| One-off question about a topic | No | Direct prompt is faster; no repeatability needed |
| Recurring structured research task | Yes | Skill ensures consistent output format every time |
| Multi-file synthesis with subagents | Yes | Skill body specifies the delegation pattern explicitly |
| Quick Q&A from a single user | No | Overhead not worth it for a single exchange |
| Automated pipeline triggered many times | Yes | Trigger makes invocation explicit and auditable |
| Onboarding checklist run by multiple people | Yes | Skill locks in the steps so they are never skipped |

> A skill is most valuable when:
> 1. The same complex workflow runs repeatedly
> 2. You need consistent output structure across runs
> 3. Subagent delegation or multi-step tool use is required

### Exercise 1 -- Write a Code-Reviewer SKILL.md

**Problem:** Write a second `SKILL.md` for a `code-reviewer` skill that:

- Trigger: `@code-review`
- Takes a GitHub URL as input (the file to review)
- Instructs the LLM to fetch the file at the URL, analyze for security issues
  (injection, hardcoded secrets, auth bypasses), and return findings as a
  markdown table: `Issue | Severity | Line | Recommendation`
- Output artifact: `security-review.md`

Fill in the TODO block in the cell below, then run the answer-key cell to check.

In [ ]:
# Exercise 1: Write a SKILL.md for a code-reviewer skill
# TODO: Fill in the YAML front-matter and markdown body

CODE_REVIEWER_SKILL = """---
# TODO: fill in these fields
name: ???
trigger: ???
description: >
  TODO: write a one-sentence description
---

# TODO: write the skill body (system prompt)
# It should instruct the LLM to:
# 1. Fetch the file at the GitHub URL provided in the query
# 2. Analyze for security issues
# 3. Return a markdown table: Issue | Severity | Line | Recommendation
# 4. Write output to security-review.md
"""

print("Your SKILL.md:")
print(CODE_REVIEWER_SKILL)

# When done, uncomment these lines to validate:
# meta_test = parse_skill_frontmatter(CODE_REVIEWER_SKILL)
# print("Parsed fields:", meta_test)

In [ ]:
# Exercise 1 -- Answer Key

CODE_REVIEWER_ANSWER = """---
name: code-reviewer
trigger: \"@code-review\"
description: >
  Fetch a source file from a GitHub URL and return a security review table.
---

# Code Review Skill

You are a security-focused code reviewer.

When activated with `@code-review <github_url>`:

1. Fetch the raw file content from the provided GitHub URL using web_fetch.
2. Analyze the code for common security issues:
   - Hardcoded secrets or API keys
   - SQL injection vulnerabilities (string interpolation in queries)
   - Authentication bypasses (missing auth checks)
   - Input validation gaps (unsanitized user input)
   - Insecure dependencies or imports
3. Return a markdown table with columns: Issue | Severity | Line | Recommendation
   - Severity levels: CRITICAL, HIGH, MEDIUM, LOW
4. If no issues are found, write \"No security issues detected.\"
5. Write the final review as an artifact named `security-review.md`.
"""

meta_review = parse_skill_frontmatter(CODE_REVIEWER_ANSWER)
body_review = extract_skill_body(CODE_REVIEWER_ANSWER)

print("Code-reviewer skill front-matter:")
for k, v in meta_review.items():
    print(f"  {k:<14} = {v!r}")
print(f"\nBody ({len(body_review)} chars):")
print(body_review[:220] + "...")
print()
print("To use: @code-review https://raw.githubusercontent.com/user/repo/main/src/auth.py")

### Exercise 2 -- Write a Trigger Parser Function

**Problem:** Write `parse_trigger(message)` that:

- Returns `(skill_name, query)` if the message starts with an `@skill-name` token
- Returns `(None, message)` if no trigger is found
- Handles edge cases: leading/trailing spaces, trigger at end with no query,
  multiple internal spaces

**Constraint:** do NOT use `re`. Use `str.split()` only.

The `skill_name` returned should be the trigger **without** the leading `@`.

In [ ]:
# Exercise 2: Trigger parser
# TODO: implement parse_trigger(message) using only str.split()

def parse_trigger(message):
    """
    Returns (skill_name, query) if message starts with @-token.
    Returns (None, message) if no trigger found.

    Rules:
    - Trigger must be the first non-whitespace token
    - Trigger starts with @
    - skill_name is the trigger without the @ prefix
    - query is everything after the trigger (stripped)
    """
    pass  # TODO


tests = [
    ("@course-research Generate a report.",  ("course-research", "Generate a report.")),
    ("@code-review https://github.com/...",  ("code-review", "https://github.com/...")),
    ("What is the reactive loop?",           (None, "What is the reactive loop?")),
    ("  @course-research  Trimmed spaces  ", ("course-research", "Trimmed spaces")),
    ("@course-research",                     ("course-research", "")),
]

print("Tests (will all show FAIL until you implement parse_trigger):\n")
for msg, expected in tests:
    result = parse_trigger(msg)
    status = "PASS" if result == expected else f"FAIL (got {result!r})"
    print(f"  [{status}] {msg!r[:55]}")

In [ ]:
# Exercise 2 -- Answer Key

def parse_trigger(message):  # noqa: F811
    tokens = message.strip().split()
    if not tokens:
        return None, message.strip()
    first = tokens[0]
    if first.startswith("@"):
        skill_name = first[1:]  # remove the @ prefix
        query = " ".join(tokens[1:]).strip()
        return skill_name, query
    return None, message.strip()


tests = [
    ("@course-research Generate a report.",  ("course-research", "Generate a report.")),
    ("@code-review https://github.com/...",  ("code-review", "https://github.com/...")),
    ("What is the reactive loop?",           (None, "What is the reactive loop?")),
    ("  @course-research  Trimmed spaces  ", ("course-research", "Trimmed spaces")),
    ("@course-research",                     ("course-research", "")),
]

all_pass = True
for msg, expected in tests:
    result = parse_trigger(msg)
    status = "PASS" if result == expected else f"FAIL (got {result!r}, expected {expected!r})"
    if "FAIL" in status:
        all_pass = False
    print(f"  [{status}] {msg!r[:55]}")

print()
print("All tests passed!" if all_pass else "Some tests failed -- check implementation.")
print()
print("Key insight: parse_trigger() mirrors what DeerFlow's dispatcher does.")
print("The real dispatcher also checks against the registry to validate the skill exists.")

### A6 -- Source Files That Part B Will Upload

The `course-research` skill reads uploaded markdown files from the thread
workspace. Part B will upload three files before sending the `@course-research`
prompt. Here is what each file contains:

| File | Pattern | Key idea |
|------|---------|----------|
| `01-state-graph.md` | State Graph | Explicit directed graph; nodes process state; edges route flow |
| `02-reactive-loop.md` | Reactive Loop | Agent calls tools in a loop until a stop condition is met |
| `03-multi-agent.md` | Multi-Agent | Supervisor delegates sub-tasks to specialist worker agents |

These are the same files defined in `src/tools.py` as `SOURCE_FILES`.

In [ ]:
# Preview the source files that will be uploaded in Part B

SOURCE_FILES = {
    "01-state-graph.md": (
        "# State Graph Pattern\n"
        "Explicit directed graph. Nodes process state; edges route flow.\n"
        "Use when: pipeline has well-defined stages with predictable transitions.\n"
        "Trade-offs: verbose setup; great for auditability and replay.\n"
    ),
    "02-reactive-loop.md": (
        "# Reactive Loop Pattern\n"
        "Agent calls tools in a loop until a stop condition is met.\n"
        "Use when: task is open-ended; number of steps is unknown upfront.\n"
        "Trade-offs: harder to debug; powerful for code generation and research.\n"
    ),
    "03-multi-agent.md": (
        "# Multi-Agent Pattern\n"
        "Supervisor delegates sub-tasks to specialist worker agents.\n"
        "Use when: task can be parallelised or needs diverse domain expertise.\n"
        "Trade-offs: coordination overhead; scales well for deep research tasks.\n"
    ),
}

print("Source files to upload in Part B:\n")
for filename, content in SOURCE_FILES.items():
    first_line = content.splitlines()[0]
    print(f"  {filename:<25}  ({len(content)} bytes)  {first_line!r}")

print()
print("Total bytes:", sum(len(c) for c in SOURCE_FILES.values()))

## Part B -- Live Research Skill Run

**Requirements before running Part B:**

1. DeerFlow server running at `http://localhost:8000`
2. The `course-research` skill registered (copy runtime files from this example)

```bash
# Clone DeerFlow next to this repo
git clone https://github.com/bytedance/deer-flow ../deer-flow
cd ../deer-flow && pip install -e .

# Copy this example's runtime config and skills
cp examples/135-deerflow-research-skill/runtime/config.sample.yaml \
   ../deer-flow/conf/config.yaml
cp -r examples/135-deerflow-research-skill/runtime/skills \
   ../deer-flow/

# Start the server
make dev
```

Cell `d023` will raise a `RuntimeError` immediately if DeerFlow is not reachable.
Part A cells are unaffected.

In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field
from typing import Iterator

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL  = os.getenv("DEERFLOW_BASE_URL", "http://localhost:8000")
THREAD_ID = "notebook-thread-135"

try:
    r = httpx.get(f"{BASE_URL}/api/health", timeout=5)
    r.raise_for_status()
    print(f"DeerFlow reachable at {BASE_URL}")
except Exception as exc:
    raise RuntimeError(
        f"DeerFlow is not reachable at {BASE_URL}.\n"
        "Start it with `make dev` in the deer-flow repo.\n"
        "Also copy runtime/config.sample.yaml and runtime/skills/ -- see Part B header."
    ) from exc


@dataclass
class ResearchRun:
    """Thin HTTP client for a DeerFlow instance configured with a custom skill."""

    base_url: str
    thread_id: str
    _http: httpx.Client = field(
        default_factory=lambda: httpx.Client(
            timeout=httpx.Timeout(60.0, read=300.0)
        )
    )

    def upload(self, filename: str, content: str) -> str:
        """Upload a markdown file to the current thread workspace."""
        resp = self._http.post(
            f"{self.base_url}/api/files/upload",
            files={"file": (filename, content.encode(), "text/markdown")},
            data={"thread_id": self.thread_id},
        )
        resp.raise_for_status()
        return resp.json().get("artifact_id", "unknown")

    def stream(
        self,
        prompt: str,
        *,
        plan_mode: bool = True,
        subagent_enabled: bool = True,
    ) -> Iterator[tuple[str, dict]]:
        """Send a prompt and yield (event_type, data) tuples from the SSE stream."""
        with self._http.stream(
            "POST",
            f"{self.base_url}/api/chat/stream",
            json={
                "message": prompt,
                "thread_id": self.thread_id,
                "plan_mode": plan_mode,
                "subagent_enabled": subagent_enabled,
            },
        ) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line.startswith("data:"):
                    continue
                raw = line.removeprefix("data:").strip()
                if raw == "[DONE]":
                    yield "end", {}
                    return
                try:
                    ev = json.loads(raw)
                    yield ev.get("type", "unknown"), ev
                except json.JSONDecodeError:
                    pass


run = ResearchRun(base_url=BASE_URL, thread_id=THREAD_ID)
print(f"ResearchRun ready  thread_id={THREAD_ID}")

In [ ]:
# Upload source files, then invoke the @course-research skill

print("Uploading source files...")
for filename, content in SOURCE_FILES.items():
    artifact_id = run.upload(filename, content)
    print(f"  {filename:<25}  artifact_id={artifact_id}")

PROMPT = (
    "@course-research Generate a structured research report: "
    "executive summary, pattern comparison table, implementation notes, "
    "and next-step recommendations."
)

print(f"\nSending prompt: {PROMPT!r}\n")
print("Streaming events:\n")

live_events = []

_LIVE_ICONS = {
    "tool_call":     "[tool]  ",
    "tool_result":   "[result]",
    "message_chunk": "[chunk] ",
    "end":           "[end]   ",
}

for et, data in run.stream(PROMPT, plan_mode=True, subagent_enabled=True):
    live_events.append((et, data))
    icon = _LIVE_ICONS.get(et, f"[{et[:6]}]")
    if et == "tool_call":
        tool_name = data.get("tool_name", "?")
        args_keys = list(data.get("args", {}).keys())
        print(f"  {icon} {tool_name}  args={args_keys}")
    elif et == "tool_result":
        content = str(data.get("content", ""))[:70]
        print(f"  {icon} {content!r}")
    elif et == "message_chunk":
        chunk = str(data.get("content", ""))[:70]
        print(f"  {icon} {chunk!r}")
    elif et == "end":
        print(f"  {icon}")

print(f"\nTotal events received: {len(live_events)}")

In [ ]:
# Summarise the event stream and look for the output artifact

counts = {}
for et, _ in live_events:
    counts[et] = counts.get(et, 0) + 1

print("Event counts:")
for et, n in sorted(counts.items()):
    print(f"  {et:<18} {n}")

print()

# Look for tool_result events that mention a markdown artifact
artifact_mentions = []
for et, data in live_events:
    if et == "tool_result":
        content = str(data.get("content", ""))
        if ".md" in content or "artifact" in content.lower():
            artifact_mentions.append(content[:120])

if artifact_mentions:
    print("Artifact references found in tool_result events:")
    for ref in artifact_mentions:
        print(f"  {ref!r}")
else:
    print("No artifact references found -- the run may still be writing the file.")

print()
print(f"Thread  : {THREAD_ID}")
print(f"Server  : {BASE_URL}")
print(f"Skill   : course-research")

## What is Next

- **DeerFlow GitHub**: https://github.com/bytedance/deer-flow -- full documentation
  on the SKILL.md format, `conf/config.yaml`, and the complete list of built-in
  tools available inside skill bodies (`web_search`, `file_read`, `file_write`,
  `code_interpreter`, and more)

- **Skill registry docs**: check DeerFlow's `conf/config.yaml` for the
  `skills.custom` format; each path entry must contain a `SKILL.md` and have
  `enabled: true`

- **Previous example**: `examples/134-deerflow-embedded-client/` -- the raw HTTP
  API and SSE streaming protocol that the skill system runs on top of; reading
  that example first gives context for the `stream()` method used in Part B

- **Example 99 in this repo**: browser agent pattern -- an alternative research
  workflow driven by tool use rather than a skill prompt; useful for comparing
  the two approaches on the same research task

- **Adapting the SKILL.md pattern**: the `@`-trigger + injected system prompt
  approach works in other frameworks too -- OpenAI Assistants (via `instructions`
  override), Anthropic tool use with preamble injection, and any framework that
  supports per-request system prompt overrides